Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel $\rightarrow$ Restart) and then **run all cells** (in the menubar, select Cell $\rightarrow$ Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [ ]:
NAME = "Georgia Takatzoglou"
ID = "230"

---

# Introduction & Learning Goals

Welcome to your second assignment! The goal is to get hands-on experience with two fundamental ways of modeling language: i) N-gram Language Models (LMs) that capture the probability of a sequence of words, ii) Word Embeddings: Using dense vectors (fasttext) to represent word meaning.

You will first build two separate n-gram LMs—one for positive reviews and one for negative—and use perplexity to see how well they model new sentences. Then, you will use word embeddings to build a sentiment classifier.

This assignment will test your understanding of the concepts from the Speech and Language Processing book, Chapters 3 (N-gram Language Models) and 5 (Vector Semantics and Embeddings).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.util import pad_sequence, ngrams
from nltk.lm.preprocessing import padded_everygram_pipeline, flatten
from nltk.lm import Laplace, MLE
import re
import requests, zipfile, io
import nltk
from datasets import load_dataset

# Download NLTK resources
nltk.download('punkt_tab')
nltk.download('stopwords')

# Load the IMDB dataset from Hugging Face
print("Loading IMDB dataset from Hugging Face...")
dataset = load_dataset('imdb')

# The dataset is a DatasetDict. We'll convert the 'train' and 'test' splits to pandas
X_train_df = dataset['train'].to_pandas()
X_test_df = dataset['test'].to_pandas()

print(f"Loaded {len(X_train_df)} training examples and {len(X_test_df)} test examples.")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Loading IMDB dataset from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

...

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Loaded 25000 training examples and 25000 test examples.


# Part 1 - N-gram Language Models

In this section, you will use nltk to build, train, and evaluate n-gram language models. You will create two separate models, one  trained only on positive reviews, and one trained only on negative reviews.

You will then use perplexity to measure how "surprising" a new sentence is to each model. A lower perplexity score means the model finds the sentence more probable (i.e., it "fits" the model better).

We will use trigrams (n=3) and Laplace (Add-1) smoothing.

## Task 1.1 Train an N-gram LM

Create a function train_lm that takes a list of tokenized sentences (a list of lists of tokens) and returns a trained 3-gram nltk language model with Laplace smoothing (nltk.lm.Laplace). Use the padded_everygram_pipeline function to process the sentences.

In [2]:
def train_lm(sentences):
    n = 3

    train_data, vocab = padded_everygram_pipeline(n, sentences)

    model = Laplace(n)

    model.fit(train_data, vocab)

    return model

In [3]:
"""Testing that the function returns the correct results"""
# Convert the train split of the DatasetDict to pandas
X_train_df = dataset['train'].to_pandas()

def preprocess(text):
    text = text.lower()
    return [word_tokenize(sent) for sent in sent_tokenize(text)]

print("Preparing data for training N-gram LMs...")
# Get all positive and negative reviews and create a list of sentences, each
# with a list of tokens suitable for nltk LM training
pos_sents = [sent for review in X_train_df[X_train_df['label'] == 1]['text'] for sent in preprocess(review)]
neg_sents = [sent for review in X_train_df[X_train_df['label'] == 0]['text'] for sent in preprocess(review)]
print(f"Total positive sentences for LM: {len(pos_sents)}")
print(f"Total negative sentences for LM: {len(neg_sents)}")
print(f"Example positive sentence: {pos_sents[0]}")
print(f"Example negative sentence: {neg_sents[0]}")

print("Training positive language model")
pos_lm = train_lm(pos_sents)
print("Training negative language model")
neg_lm = train_lm(neg_sents)

assert pos_lm.counts[['great']]['movie'] == 294
assert neg_lm.counts[['bad']]['movie'] == 320


Preparing data for training N-gram LMs...
Total positive sentences for LM: 130829
Total negative sentences for LM: 137359
Example positive sentence: ['zentropa', 'has', 'much', 'in', 'common', 'with', 'the', 'third', 'man', ',', 'another', 'noir-like', 'film', 'set', 'among', 'the', 'rubble', 'of', 'postwar', 'europe', '.']
Example negative sentence: ['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was', 'first', 'released', 'in', '1967.', 'i', 'also', 'heard', 'that', 'at', 'first', 'it', 'was', 'seized', 'by', 'u.s.', 'customs', 'if', 'it', 'ever', 'tried', 'to', 'enter', 'this', 'country', ',', 'therefore', 'being', 'a', 'fan', 'of', 'films', 'considered', '``', 'controversial', "''", 'i', 'really', 'had', 'to', 'see', 'this', 'for', 'myself.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'the', 'plot', 'is', 'centered', 'around', 'a', 'young', 'swedish', 'drama', 'stude

## Task 1.2 Calculate Perplexity

Create a calculate perplexity function that takes a trained model and an untokenized sentence and computes its perplexity. You can use the perplexity function of nltk models. Remember to apply the same preprocessing as during training.

In [4]:
def calculate_perplexity(lm, test_sentence):
    test_sentence = test_sentence.lower()
    tokens = word_tokenize(test_sentence)

    n = 3
    padded = list(pad_sequence(tokens, n,
                               pad_left=True, left_pad_symbol="<s>",
                               pad_right=True, right_pad_symbol="</s>"))

    test_ngrams = list(ngrams(padded, n))

    return lm.perplexity(test_ngrams)

In [5]:
"""Testing that the function returns the correct results"""
test_pos_sent = "This was a truly great and wonderful film."
per_pos_pos = calculate_perplexity(pos_lm, test_pos_sent)
per_neg_pos = calculate_perplexity(neg_lm, test_pos_sent)
assert per_pos_pos < per_neg_pos


# Part 2 - Embeddings

In this section, we'll switch gears. Instead of n-grams, we'll represent text using pre-trained fasttext embeddings. We will use a "sentence embedding" technique by averaging the word vectors for all words in a sentence to build a sentiment classifier.

In [6]:
!pip install fasttext
import fasttext
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="facebook/fasttext-en-vectors", filename="model.bin")
model = fasttext.load_model(model_path)
#model = fasttext.load_model("model.bin")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.1-py3-none-any.whl.metadata (10.0 kB)
Using cached pybind11-3.0.1-py3-none-any.whl (293 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4498215 sha256=c1eaef4e11bd86b1b73a60e7a2798efffaf308150ce19c7477cde01ee0f61a49
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


model.bin:   0%|          | 0.00/7.24G [00:00<?, ?B/s]

## Task 2.1 Implement Cosine Similarity

To understand embeddings, let's look at their properties. A key operation is cosine similarity, which measures the similarity between two vectors. Implement the function below.Hint: The formula is $similarity = \frac{A \cdot B}{\|A\| \|B\|}$

In [7]:
def cosine_similarity(vec_a, vec_b):
    dot = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    return dot / (norm_a * norm_b)

In [8]:
"""Testing that the function returns the correct results"""
vec_good = model['good']
vec_nice = model['nice']
vec_king = model['king']
assert cosine_similarity(vec_good, vec_nice) > cosine_similarity(vec_good, vec_king)


## Task 2.2 Document Vector Averaging

To create a sentiment classifier, let's first implement a method that converts a document into the average of the embeddings of the words inside the document:

In [12]:
def average_document_vector(doc_text, embeddings_dict):
    tokens = word_tokenize(doc_text.lower())

    vectors = []

    for w in tokens:
        try:
            vectors.append(embeddings_dict[w])
        except KeyError:
            pass
    if not vectors:
        return np.zeros(embeddings_dict.get_dimension())

    return np.mean(vectors, axis=0)

In [13]:
"""Testing that the function returns the correct results"""
from datasets import concatenate_datasets

train_dataset = load_dataset("imdb", split="train")
positive_samples = train_dataset.filter(lambda x: x["label"] == 1)
negative_samples = train_dataset.filter(lambda x: x["label"] == 0)
n = 500
positive_subsample = positive_samples.select(range(n))
negative_subsample = negative_samples.select(range(n))
subsampled_dataset = concatenate_datasets([positive_subsample, negative_subsample])

texts = subsampled_dataset['text']
labels = subsampled_dataset['label']

# 2. Convert text documents to averaged vector features
print(f"Converting {len(texts)} documents to averaged fasttext vectors...")
X = np.array([average_document_vector(text, model) for text in texts])
y = np.array(labels)

# Check the shape of the features
print(f"Feature matrix shape (X): {X.shape}")
print(f"Label vector shape (y): {y.shape}")

# 3. Split data into training and testing sets
#X_train, X_test, y_train, y_test = train_test_split(
#    X, y, test_size=0.2, random_state=42, stratify=y
#)
#print(f"Train/Test split: {len(X_train)} training samples, {len(X_test)} testing samples.")

# 4. Train the Logistic Regression Model
print("Training Logistic Regression model...")
classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X, y)
print("Training complete.")

assert classifier.predict([average_document_vector("This was a truly great and wonderful film.", model)])[0] == 1

Converting 1000 documents to averaged fasttext vectors...
Feature matrix shape (X): (1000, 300)
Label vector shape (y): (1000,)
Training Logistic Regression model...
Training complete.
